# 68b · ConvNeXtV2 ở **1022** — giả thuyết: CNN tận dụng độ phân giải tốt hơn ViT

## Bằng chứng dẫn tới notebook này

`nb68` chạy RAD-DINO (ViT-B/14) ở `1022`, fine-tune backbone ở đúng độ phân giải đó:

```
             64 @518   68 @1022    chenh
macro         0.2645     0.2590   -0.0055
head          0.5593     0.5545   -0.0048
medium        0.2227     0.2256   +0.0029   <- LEN
tail          0.1379     0.1241   -0.0138   <- XUONG manh
```

Mọi nhãn hại nặng đều là **tail**; mọi nhãn lợi đều là **medium**. Cơ chế: `40` query lớp phải
học phân bố attention trên `5,329` khoá thay vì `1,369` — **pha loãng thống kê**, và nhãn có `129`
ca dương trả giá nặng nhất.

## Vì sao CNN có thể khác hẳn

```
ViT-B/14  luoi PHANG, attention toan cuc
          them pixel = them KHOA ma 40 query phai HOC cach phan bo len
          518 -> 1,369 khoa    |   1022 -> 5,329 khoa

ConvNeXt  PHAN CAP stride 4/8/16/32, tich chap CHIA SE trong so theo vi tri
          them pixel = them MAU KHONG GIAN cho CUNG bo kernel — so tham so KHONG doi
          518 -> 16x16 = 256 khoa   |   1022 -> 31x31 = 961 khoa   (da do bang timm)
```

Ở `1022`, K/V của ML-Decoder là **961** thay vì **5,329** — pha loãng nhẹ hơn `5.5×`. Và chi tiết
mảnh vẫn được xử lý ở tầng dưới:

```
1022px, 0.342 mm/px:  stride 4 -> 1.37mm | 8 -> 2.74mm | 16 -> 5.48mm | 32 -> 10.96mm
 518px, 0.676 mm/px:  stride 4 -> 2.70mm | ...                        | 32 -> 21.63mm
```

Đường gãy xương đòn `1–2mm` **nằm trong tầm stage-1 ở 1022** (`1.37mm`), không nằm trong tầm ở 518.

## Bảng xếp hạng của giải nói cùng một chuyện

```
doi C  ConvNeXt-S + EfficientNetV2-S   1024   CNN
doi D  ConvNeXt-S                      1024   CNN
doi B  EfficientNetV2-L                 512   CNN
doi A  ConvNeXt x4                 224, 384   CNN
doi E  ViT-L (DINOv2, 710k anh CXR)  336/448/512   <- ViT, do phan giai THAP
doi G  ViT-L                            224        <- ViT, THAP nhat bang
```

**Mọi đội dùng `1024` đều dùng CNN. Cả hai đội dùng ViT-L đều ở `≤512`.** Không đội nào chạy ViT ở
`1024`.

## Khác `64b` ở đâu

```
64b                              68b (day)
SIZE 518                         SIZE 1022, doc THANG shard goc (khong resize lan nao)
hydrate tu cxrlt_518_ready       kho do CHI co ban 518 -> stage tu shard `cxr_1022_letterbox_cxrlt`
BATCH 48 x ACCUM 6 = 288         BATCH 12 x ACCUM 24 = 288  (giu nguyen EFF)
E=8, 2 seed                      CHE_DO_THU: E=6, 1 seed  (so voi 64b tai ep6)
TIER = INV['tier'] (13/16/11)    TIER = dinh nghia CHINH THUC cua giai (9/15/16)
run r2a -> stage4_nb64b          run r1022 -> stage4_nb68b
```

Backbone khởi tạo vẫn là `convnextv2_padchest.pt` — ConvNeXt **toàn tích chập** nên nhận mọi kích
thước, không cần nội suy pos-embed như ViT.

## Cổng khoá trước

```
1. tail KHONG tut qua -0.003 so voi 64b @518 (ViT tut -0.0138 — day la thu can bac bo)
2. 7 nhan medium tang >= +0.01 TB
3. macro >= 64b @518 - 0.0007
```

Đạt cả ba ⇒ chạy đầy đủ (`E=8`, 2 seed) rồi đưa vào ensemble.

## Chi phí

`64b` đo `43.9 ph/epoch` @518. ConvNeXt là toàn tích chập ⇒ chi phí **tuyến tính theo pixel**
(`3.89×`), không bậc hai như attention.

```
CHE_DO_THU  E=6, 1 seed   ~171 ph/epoch  ->  ~17h
day du      E=8, 2 seed                  ->  ~46h
```

Đĩa: ảnh `1022` chiếm `~49 GB`. Dùng chung `/content/cxrlt_1022` với `nb68` nếu cùng session.


## C0 · Config

In [1]:
import os, json, math, time, re, glob, datetime, shutil, tarfile, io as _io
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
try:
    import timm
except ImportError:
    import subprocess, sys as _s
    subprocess.run([_s.executable, '-m', 'pip', 'install', '-q', 'timm']); import timm
try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = os.environ.get('CXRLT_DATA', '')
    assert DATA_DIR, 'khong phai Colab va chua dat CXRLT_DATA -> gan thang duong dan `planB_out`'
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024'); EMB_DIR = os.path.join(OUT_DIR, 'embeddings')

# Chan spam `Exception ignored in __del__: can only test a child process` cua DataLoader.
# No KHONG di qua `warnings` — Python bao cao qua `unraisablehook`. Loc DUNG loai nay.
import sys as _sys
_old_unraisable = _sys.unraisablehook
def _quiet_unraisable(u):
    if isinstance(u.exc_value, AssertionError) and 'child process' in str(u.exc_value): return
    _old_unraisable(u)
_sys.unraisablehook = _quiet_unraisable

# --- cau hinh DO PHAN GIAI: phai dung TRUOC RUN_TAG va LOCAL (chung dung SIZE) ---
SIZE_SHARD = 1022    # kich thuoc anh trong shard `cxr_1022_letterbox_cxrlt`
CHE_DO_THU = False   # True: ngan sach rut gon de TRA LOI CAU HOI (E=6, 1 seed)
                     # False: luot DAY DU (E=8, 2 seed) — ~46 gio GPU
SIZE, BATCH, ACCUM = 1022, 12, 24                 # EFF 288 GIU NGUYEN; batch nho vi 3.89x pixel
assert BATCH * ACCUM == 288, 'phai giu EFF 288 nhu 64b, neu khong thi khong so duoc'
print(f'DO PHAN GIAI {SIZE} | ConvNeXt toan tich chap -> khong can noi suy pos-embed')

# RUN_TAG PHAI khac giua luot THU va luot DAY DU. Hai ly do, ca hai deu tung lam hong
# du lieu trong du an:
#  1. C4 bo qua seed co co `done` ma KHONG xet no xong o may epoch. s42 da `done` o
#     E=6; giu nguyen tag thi no bi bo qua o E=8 -> hai seed lech do sau.
#  2. Ghi de cung ten file qua Drive FUSE thi chi lan ghi DAU xuong dia, ma `put()`
#     chi kiem KICH THUOC nen khong thay. Lich LR cua E=8 khac E=6 -> file ep1-6 giu
#     noi dung cu, tron lan hai cau hinh.
# Tag rieng => thu muc moi tinh, luot thu (r1022) van nguyen ven de doi chieu.
RUN_TAG = f'r{SIZE}' if CHE_DO_THU else f'r{SIZE}_E8'
# Tach theo RUN_TAG: neu chay luot thu roi luot day du trong CUNG mot session, nhat ky
# local cua luot truoc se duoc `find()` bat truoc (SEARCH = [RES_LOC, RES41D]) va s42
# lai bi coi la `done`.
RES_LOC = f'/content/nb41e_{RUN_TAG}'; os.makedirs(RES_LOC, exist_ok=True)   # NOI GHI CHINH
RES41D  = os.path.join(OUT_DIR, 'stage4_nb68b', RUN_TAG)   # giu ten bien de khoi dung cham noi khac          # ban sao tren Drive
RES41C  = os.path.join(OUT_DIR, 'stage4_nb41c')
SEARCH  = [RES_LOC, RES41D]
def ensure_res():
    try: os.makedirs(RES41D, exist_ok=True)
    except Exception: pass                       # mount hong -> ke, /content moi la noi ghi chinh
    return RES41D
PUT_FAIL = []                                   # ten file KHONG len duoc Drive
def put(name, fn):
    '''Ghi LOCAL truoc (chac chan), roi sao len Drive va XAC MINH KICH THUOC.
       Lan chay truoc mat sach vi ban cu chi IN canh bao roi di tiep — xem `drive_gate()`.'''
    lp = os.path.join(RES_LOC, name); fn(lp)
    dp = os.path.join(RES41D, name)
    try:
        ensure_res(); shutil.copy(lp, dp)
        sz_l, sz_d = os.path.getsize(lp), os.path.getsize(dp)
        if sz_d != sz_l: raise OSError(f'kich thuoc lech {sz_d} != {sz_l}')
    except Exception as e:
        PUT_FAIL.append(name)
        print(f'  !! sao {name} len Drive THAT BAI ({type(e).__name__}: {e})')
    return lp

def drive_probe(mb=8):
    '''Ghi THAT roi doc lai — `os.path.exists` KHONG du: FUSE cho thay file rong/ma.
       130 GB trong ma ghi 100 MB van MAT la ca ta da gap.'''
    ensure_res()
    p = os.path.join(RES41D, '_probe.bin'); blob = os.urandom(mb * 1024 * 1024)
    try:
        with open(p, 'wb') as f: f.write(blob); f.flush(); os.fsync(f.fileno())
        with open(p, 'rb') as f: back = f.read()
        os.remove(p)
        return back == blob
    except Exception:
        return False

def drive_gate(ep_tag):
    '''Goi CUOI moi epoch. Drive hong -> DUNG NGAY thay vi chay tiep 10 gio roi mat sach.'''
    if not PUT_FAIL: return
    raise AssertionError(
        f'{len(PUT_FAIL)} file KHONG len duoc Drive o {ep_tag}: {PUT_FAIL[:4]}\n'
        f'  Hien vat CON o {RES_LOC} — dung tat session truoc khi cuu.\n'
        '  Cach cuu: (1) chay `!cp -r ' + RES_LOC + ' /content/drive/MyDrive/_cuu41e` thu xem;\n'
        '  (2) neu that bai -> Runtime > Restart, mount lai, chay C0-C1 roi C4 (co resume);\n'
        '  KHONG xoa content_cache — DriveFS giu ghi cho o do.')
def find(pat):
    seen = {}
    for d in SEARCH:
        for f in sorted(glob.glob(os.path.join(d, pat))): seen.setdefault(os.path.basename(f), f)
    return [seen[k] for k in sorted(seen)]
ensure_res()
assert drive_probe(), (
    'Drive KHONG ghi-doc lai duoc (probe 8 MB that bai) -> mount hong.\n'
    '  Lan chay 41e truoc mat SACH 10 gio GPU vi loi nay khong duoc chan o day.\n'
    '  Sua: Runtime > Restart runtime, chay lai tu C0. KHONG xoa content_cache.')
print('cong 0: Drive ghi-doc lai duoc (probe 8 MB)')
LOCAL = f'/content/cxrlt_{SIZE}'
READY_X = os.path.join(OUT_DIR, 'cxrlt_518_ready')

# (SRC_PIPE/ARM_PIPE giu lai cho tuong thich; 64b KHONG con nap duong ong)
SRC_PIPE = [os.path.join(OUT_DIR, 'stage4_nb57', 'r3'),
            os.path.join(OUT_DIR, 'stage4_nb53', 'r3')]
ARM_PIPE = ('tpb', 't1_prior_both')
ARM_IMG  = 'mldec'                               # arm ANH-THUAN -> corr backbone thuan

MODEL = 'convnextv2_nano.fcmae_ft_in22k_in1k'
P_BB  = os.path.join(RES41C, 'convnextv2_padchest.pt')
P_OOF = os.path.join(OUT_DIR, 'stage4_nb52a', 'xgb_oof.npy')
SEED, SEEDS = 42, ((42,) if CHE_DO_THU else (42, 43))   # che do thu: 1 seed
EPOCHS = 6 if CHE_DO_THU else 8                   # so voi 64b tai CUNG epoch
LR, WD, WARMUP, CLIP = 1e-4, 0.05, 0.05, 5.0      # y het 64b
# --- NGAN SACH RAM cho DataLoader: ti le voi PIXEL, KHONG voi kich thuoc model ---
# Model 15M tham so nam o VRAM (60 MB). RAM he thong bi an boi TENSOR ANH DA GIAI MA
# trong hang doi prefetch cua worker:   RAM = bs x 2 anh x MB_anh x NW x prefetch
#   518px  ->  3.22 MB/anh    1022px -> 12.53 MB/anh   (x3.9)
# Cau hinh cu (NW 16, prefetch 4, bs_eval 128) ton 52 GB o 518 nhung 205 GB o 1022 -> TRAN.
_MB_ANH  = 3 * SIZE * SIZE * 4 / 1e6
_LON     = SIZE > 700
NW       = min(4 if _LON else 16, (os.cpu_count() or 2))
PREFETCH = 2 if _LON else 4
BS_EVAL  = 32 if _LON else 128          # dung cho predict(); KHONG doi ket qua, chi doi toc do
BS_DIAG  = 16 if _LON else 64           # lo chuan doan ratio_img
_RAM_GB  = (BATCH + 2 * BS_EVAL + BS_DIAG) * 2 * _MB_ANH * NW * PREFETCH / 1e3
print(f'DataLoader: NW {NW} | prefetch {PREFETCH} | bs_eval {BS_EVAL} | '
      f'{_MB_ANH:.1f} MB/anh -> RAM toi da ~{_RAM_GB:.0f} GB')
assert _RAM_GB < 60, (f'DataLoader se an ~{_RAM_GB:.0f} GB RAM -> ha NW/PREFETCH/BS_EVAL. '
                      'Day la RAM HE THONG (tensor anh), khong lien quan kich thuoc model.')
D_MLD, D_ETOK, EHR_CLIP = 512, 256, 10.0
MOC, GATE_BLEND = 0.8236, 0.005
DO_TEST = True
P_TOUCH = os.path.join(OUT_DIR, '_test_touches_nb68b.jsonl')

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG thay GPU'
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'thiet bi: {torch.cuda.get_device_name(0)} | '
      f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB | AMP {AMP}')

_bb_loc = '/content/convnextv2_padchest.pt'
BB_USE = P_BB if os.path.exists(P_BB) else (_bb_loc if os.path.exists(_bb_loc) else None)
assert BB_USE, f'khong thay trong so PadChest o CA {P_BB} lan {_bb_loc} -> chay 41c C3 truoc'
for p, hd in ((P_OOF, '52a'), (os.path.join(OUT_DIR, 'cxrlt_micro_full.parquet'), '29c'),
              (os.path.join(OUT_DIR, 'cxrlt_micro_vocab.json'), '29c')):
    assert os.path.exists(p), f'khong thay {p} -> chay {hd} truoc'
print(f'trong so CNN: {BB_USE}')

INV = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER_CU = {t: np.array([LAB40.index(l) for l in INV['tier'][t]])
           for t in ('head', 'medium', 'tail')}
ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
YM = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
        .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())[LAB40].to_numpy('float32')
SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'; TE = ~(TR | VA)
# TIER theo dinh nghia CHINH THUC cua giai (paper §3.3: prevalence tren TRAIN)
_prev = YM[SPLIT == 'train'].mean(0)
TIER = {'head':   np.where(_prev > 0.10)[0],
        'medium': np.where((_prev >= 0.01) & (_prev <= 0.10))[0],
        'tail':   np.where(_prev < 0.01)[0]}
print('TIER = CHINH THUC ' + str({k: len(v) for k, v in TIER.items()})
      + ' | TIER_CU (nb28) ' + str({k: len(v) for k, v in TIER_CU.items()}))
_rs = np.random.default_rng(SEED); _su = np.unique(SUB[TR])
_sel = _rs.choice(_su, size=int(round(0.15*len(_su))), replace=False)
SL = TR & np.isin(SUB, _sel)     # van tinh, CHI de bao cao bao nhieu anh duoc thu hoi
TF_ = TR.copy()                   # <<< R2a: GOP lat chon vao train
print(f'R2a: thu hoi {int(SL.sum()):,} anh lat chon vao train '
      f'({int((TR & ~SL).sum()):,} -> {int(TF_.sum()):,}, +{SL.sum()/max((TR&~SL).sum(),1):.1%})')
print('     LAT CHON GIO NAM TRONG TRAIN -> khong duoc chon bat cu thu gi o luot nay.')
# R2a: SL GIO NAM TRONG TF_ nen giao cua chung DUNG BANG SL — kiem THUAN chieu do.
assert int((TF_ & SL).sum()) == int(SL.sum()), 'lat chon CHUA duoc gop het vao train'
assert int(TF_.sum()) == int(TR.sum()), 'TF_ phai bang DUNG TR o luot R2a'
assert not (set(SUB[TR]) & set(SUB[VA])), 'benh nhan train ro ri sang VAL'
assert not (set(SUB[TR]) & set(SUB[TE])), 'benh nhan train ro ri sang TEST'
I_TF, I_SL, I_VA, I_TE = (np.where(m)[0] for m in (TF_, SL, VA, TE))
print(f'{len(ORDER):,} anh | khop {len(I_TF):,} | chon {len(I_SL):,} | val {len(I_VA):,}')
print(f'{MODEL} | E={EPOCHS} LR={LR} | BATCH {BATCH}x{ACCUM}={BATCH*ACCUM} | seeds {SEEDS}')

Mounted at /content/drive
DO PHAN GIAI 1022 | ConvNeXt toan tich chap -> khong can noi suy pos-embed
cong 0: Drive ghi-doc lai duoc (probe 8 MB)
DataLoader: NW 4 | prefetch 2 | bs_eval 32 | 12.5 MB/anh -> RAM toi da ~18 GB
thiet bi: NVIDIA RTX PRO 6000 Blackwell Server Edition | VRAM 102 GB | AMP torch.bfloat16
trong so CNN: /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/stage4_nb41c/convnextv2_padchest.pt
TIER = CHINH THUC {'head': 9, 'medium': 15, 'tail': 16} | TIER_CU (nb28) {'head': 13, 'medium': 16, 'tail': 11}
R2a: thu hoi 39,086 anh lat chon vao train (219,785 -> 258,871, +17.8%)
     LAT CHON GIO NAM TRONG TRAIN -> khong duoc chon bat cu thu gi o luot nay.
377,110 anh | khop 258,871 | chon 39,086 | val 39,293
convnextv2_nano.fcmae_ft_in22k_in1k | E=8 LR=0.0001 | BATCH 12x24=288 | seeds (42, 43)


In [2]:
# ==== GOM 17 thu muc trung ten r1022_E8 ve MOT — khong copy du lieu, chi doi cho ====
import glob as _g, shutil as _sh
GOC = os.path.join(OUT_DIR, 'stage4_nb68b')
DUP = sorted(d for d in _g.glob(os.path.join(GOC, RUN_TAG + '*')) if os.path.isdir(d))
CANON = RES41D                                              # ban khong co " (n)"
assert CANON in DUP, f'{CANON} khong ton tai'
print(f'{len(DUP)} thu muc ten {RUN_TAG}*')

moved = skipped = 0
for d in DUP:
    if d == CANON: continue
    for f in sorted(_g.glob(os.path.join(d, '*'))):
        dst = os.path.join(CANON, os.path.basename(f))
        if os.path.exists(dst):
            skipped += 1; continue                            # da co -> giu ban DAU
        _sh.move(f, dst); moved += 1
print(f'doi cho {moved} file | bo qua {skipped} trung ten')

# --- dem lai, DO THEO TEN ---
ok = True
for sd in SEEDS:
    n = {k: sum(os.path.exists(os.path.join(CANON, f'{k}_cnn_full_s{sd}_ep{e}.npy' if k != 'w'
                else f'w_s{sd}_ep{e}.pt')) for e in range(1, EPOCHS + 1)) for k in ('p', 'pte', 'w')}
    print(f'  s{sd}: p_ {n["p"]}/{EPOCHS}  pte_ {n["pte"]}/{EPOCHS}  w_ {n["w"]}/{EPOCHS}')
    ok &= all(v == EPOCHS for v in n.values())
assert ok, 'van thieu file sau khi gom -> con thu muc trung o cho khac, gui toi ten thu muc'

# --- nhat ky: dung ban day du nhat trong cac thu muc, dam bao co du 2 seed ---
_logs = [os.path.join(d, 'epoch_log.json') for d in DUP if os.path.exists(os.path.join(d, 'epoch_log.json'))]
_best = max(_logs, key=lambda p: len(json.load(open(p))))
LOG = json.load(open(_best))
assert all(f's{s}' in LOG and LOG[f's{s}'].get('done') for s in SEEDS), f'nhat ky thieu seed: {list(LOG)}'
if _best != os.path.join(CANON, 'epoch_log.json'):
    _sh.copy(_best, os.path.join(CANON, 'epoch_log.json'))
os.makedirs(RES_LOC, exist_ok=True); _sh.copy(os.path.join(CANON, 'epoch_log.json'), os.path.join(RES_LOC, 'epoch_log.json'))
print(f'nhat ky: {list(LOG)} -> chay C5')


17 thu muc ten r1022_E8*
doi cho 23 file | bo qua 7 trung ten
  s42: p_ 8/8  pte_ 8/8  w_ 8/8
  s43: p_ 8/8  pte_ 8/8  w_ 8/8
nhat ky: ['s42', 's43'] -> chay C5


## C1 · EHR + phim trước + ảnh + dự đoán đường ống

In [ ]:
SPEC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')))
E_DF = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_ehr.parquet'))
_sid = ORDER.study_id.to_numpy()
COLS = [c for c in SPEC['Q_FULL'] if c in E_DF.columns]
_m = np.nan_to_num((E_DF.set_index('study_id')[COLS].reindex(_sid)
                    .apply(pd.to_numeric, errors='coerce').to_numpy('float32')), nan=0.0)
# Cot NHI PHAN thi KHONG z-score: p = 1/219,785 -> gia tri chuan hoa 468 -> softmax bao hoa -> NaN
_b = np.array([np.isin(np.unique(_m[:, j]), (0.0, 1.0)).all() for j in range(_m.shape[1])])
_mu = _m[TF_].mean(0, keepdims=True); _sd = _m[TF_].std(0, keepdims=True) + 1e-6
_mu[:, _b] = 0.0; _sd[:, _b] = 1.0
E_ALL = np.clip((_m - _mu) / _sd, -EHR_CLIP, EHR_CLIP).astype('float32')
_keep = [i for i, c in enumerate(COLS) if not re.match(r'^micro_org_', c)]
assert len(COLS) - len(_keep) == 30, f'bo {len(COLS)-len(_keep)} cot micro_org_*, phai la 30'
EHR_COL = E_ALL[:, _keep]

_MIC = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_micro_full.parquet')).set_index('study_id')
_VOC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_micro_vocab.json')))
V_MIC = int(_VOC['V']) + 1; PAD_ID = V_MIC - 1
_mic = _MIC.reindex(_sid)
_ids = _mic.micro_org_ids.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
_cnt = _mic.micro_org_cnt.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
MIC_L = max(1, int(_ids.map(len).max())); assert MIC_L <= 64
MIDS = np.full((len(ORDER), MIC_L), PAD_ID, 'int64'); MCNT = np.zeros((len(ORDER), MIC_L), 'float32')
for r, (a, b) in enumerate(zip(_ids, _cnt)):
    if a: MIDS[r, :len(a)] = a; MCNT[r, :len(a)] = b

XGF = np.load(P_OOF).astype('float32')
assert XGF.shape == (len(ORDER), C) and np.isfinite(XGF).all()
_l = np.log(np.clip(XGF, 1e-6, 1-1e-6) / (1 - np.clip(XGF, 1e-6, 1-1e-6)))
XGB_F = np.clip((_l - _l[TF_].mean(0, keepdims=True)) / (_l[TF_].std(0, keepdims=True) + 1e-6),
                -EHR_CLIP, EHR_CLIP).astype('float32')
FEAT = np.concatenate([EHR_COL, XGB_F], 1)
print(f'FEAT {FEAT.shape} | tui vi khuan V={V_MIC-1} dai {MIC_L}')

_BASE = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_base.parquet'))
_BASE['index_time'] = pd.to_datetime(_BASE.index_time)
_B = (_BASE[_BASE.study_id.isin(set(ORDER.study_id.unique()))]
      .sort_values(['subject_id','index_time']).reset_index(drop=True))
_B['prev_study'] = _B.groupby('subject_id', sort=False).study_id.shift(1)
_PV = dict(zip(_B.study_id.to_numpy(), _B.prev_study.to_numpy()))
_rw = {}
for _i, _s in enumerate(_sid): _rw.setdefault(_s, []).append(_i)
PREV_ROW = np.full(len(ORDER), -1, 'int64')
for _i, _s in enumerate(_sid):
    _p2 = _PV.get(_s, np.nan)
    if not (isinstance(_p2, float) and np.isnan(_p2)): PREV_ROW[_i] = _rw[int(_p2)][0]
_hp = PREV_ROW >= 0
print(f'anh co phim truoc: {_hp.sum():,} / {len(ORDER):,} ({_hp.mean():.1%})  (44a do 67.2%)')
assert _hp.mean() > 0.5 and (PREV_ROW[_hp] != np.arange(len(ORDER))[_hp]).all()

from PIL import Image
os.makedirs(LOCAL, exist_ok=True)
_n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
print(f'anh dang co: {_n:,} / {len(ORDER):,}', flush=True)
from concurrent.futures import ThreadPoolExecutor
# Kho `cxrlt_518_ready` CHI co ban 518 -> o do phan giai khac phai stage tu shard GOC.
if SIZE == 518 and _n < len(ORDER) * 0.999 and os.path.isdir(READY_X):
    _kt = sorted(x for x in os.listdir(READY_X) if x.endswith('.tar'))
    print(f'kho co {len(_kt)} tar -> hydrate', flush=True)
    for _i, f in enumerate(_kt):
        src = os.path.join(READY_X, f); tmp = os.path.join('/content', f)
        try:
            shutil.copy(src, tmp)                # copy TRUOC — khong mo tar tren mount FUSE
            assert os.path.getsize(tmp) == os.path.getsize(src), 'copy NGAN tu Drive'
            with tarfile.open(tmp) as tar: tar.extractall(LOCAL, filter='data')
        except Exception as e:
            print(f'  {f} HONG ({type(e).__name__}) -> bo qua')
        if os.path.exists(tmp): os.remove(tmp)
        if (_i + 1) % 50 == 0: print(f'  hydrate {_i+1}/{len(_kt)}', flush=True)
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
if _n < len(ORDER) * 0.999:
    SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')
    assert os.path.isdir(SHARD_DIR), f'khong thay CA {READY_X} lan {SHARD_DIR}'
    TARS = sorted(os.path.join(SHARD_DIR, f) for f in os.listdir(SHARD_DIR)
                  if f.endswith('.tar') and f.startswith('lb_'))
    DONE = os.path.join(LOCAL, '_done.json')
    _done = set(json.load(open(DONE))) if os.path.exists(DONE) else set()
    def _one(pl):
        nm, blob = pl; dst = os.path.join(LOCAL, os.path.basename(nm))
        if os.path.exists(dst): return
        try:
            if SIZE == SIZE_SHARD:     # shard DA dung kich thuoc -> ghi THANG, khoi giai ma
                tmp = dst + '.tmp'     # + ma hoa lai 377k anh (tiet kiem PHAN LON thoi gian)
                with open(tmp, 'wb') as _f: _f.write(blob)
                os.replace(tmp, dst); return
            im = Image.open(_io.BytesIO(blob)).convert('L')
            if im.size != (SIZE, SIZE): im = im.resize((SIZE, SIZE), Image.BICUBIC)
        except Exception:
            return                     # mot anh hong KHONG duoc giet ca lan chay
        tmp = dst + '.tmp'; im.save(tmp, 'JPEG', quality=95); os.replace(tmp, dst)   # NGUYEN TU
    with ThreadPoolExecutor(8) as pool:
        for tp in tqdm([t for t in TARS if os.path.basename(t) not in _done],
                       desc=f'stage {SIZE}', unit='tar'):
            with open(tp, 'rb') as f: buf = f.read()   # doc BULK; lap tren mount FUSE se TREO
            assert len(buf) == os.path.getsize(tp), f'{os.path.basename(tp)}: doc NGAN tren mount'
            with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
                jobs = [(m.name, tar.extractfile(m).read())
                        for m in tar.getmembers() if m.name.endswith('.jpg')]
            list(pool.map(_one, jobs)); del buf, jobs
            _done.add(os.path.basename(tp)); json.dump(sorted(_done), open(DONE, 'w'))
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
assert _n >= len(ORDER) * 0.999, f'thieu {len(ORDER)-_n:,} anh'
print(f'anh local: {_n:,} / {len(ORDER):,}')

# 64b KHONG nap du doan duong ong nua. Cell blend cua 41e da bi bo: no quet trong so `w`
# TREN LAT CHON, ma o R2a lat chon la du lieu huan luyen -> moi diem tren duong cong deu
# la TRONG MAU. Viec tron lam o `61`, voi `w` DA KHOA (head 0.40 | medium 0.40 | tail 0),
# sau khi ca `64` va `64b` chay xong.


FEAT (377110, 257) | tui vi khuan V=305 dai 10
anh co phim truoc: 253,283 / 377,110 (67.2%)  (44a do 67.2%)
anh dang co: 0 / 377,110


stage 1022:   0%|          | 0/378 [00:00<?, ?tar/s]

## C2 · Kiến trúc — `Tier1`/`Tier1Img`/ML-Decoder verbatim `53` r3, backbone ConvNeXt

`convnextv2_nano` @518 cho `17×17 = 289` token dim 640. `forward_features` trả `[B, H, W, Cf]`
(channels-last ở ConvNeXt của `timm`) — cổng 1 kiểm hình dạng thay vì tin giả định.

In [ ]:
class GroupFC(nn.Module):
    def __init__(s, C, d):
        super().__init__()
        s.w = nn.Parameter(torch.randn(C, d) * d ** -0.5); s.b = nn.Parameter(torch.zeros(C))
    def forward(s, h): return (h * s.w).sum(-1) + s.b

class XAttn(nn.Module):
    def __init__(s, d, nhead=8, ff=2048, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff, d))
    def forward(s, q, kv, need_w=False):
        a, w = s.at(q, kv, kv, need_weights=need_w, average_attn_weights=True)
        h = s.n1(q + a); return s.n2(h + s.ff(h)), w

class MicroBag(nn.Module):
    def __init__(s, V, d):
        super().__init__()
        s.w = nn.Parameter(torch.randn(V, d) * d ** -0.5)
        s.mlp = nn.Sequential(nn.GELU(), nn.Linear(d, d))
    def forward(s, ids, w): return s.mlp(F.embedding_bag(ids, s.w, per_sample_weights=w, mode='sum'))

class ColTokBag(nn.Module):
    def __init__(s, d_in, d=256, V=None):
        super().__init__()
        s.W = nn.Parameter(torch.randn(d_in, d) * d ** -0.5)
        s.b = nn.Parameter(torch.zeros(d_in, d))
        s.joint = nn.Sequential(nn.Linear(d_in, d), nn.GELU(), nn.Linear(d, d))
        s.ln = nn.LayerNorm(d); s.mb = MicroBag(V, d) if V else None
        if s.mb is not None: s.bln = nn.LayerNorm(d)
    def forward(s, e, mids=None, mcnt=None):
        col = s.b.unsqueeze(0) + e.unsqueeze(-1) * s.W.unsqueeze(0)
        base = s.ln(torch.cat([col, s.joint(e).unsqueeze(1)], 1))
        if s.mb is None: return base
        return torch.cat([base, s.bln(s.mb(mids, mcnt)).unsqueeze(1)], 1)

class Tier1(nn.Module):
    '''Q = patch, K/V = token EHR. `pr` zero-init; `ln_a` chuan hoa DAU VAO cua phep chieu (38d).'''
    def __init__(s, d, nhead=8, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d); s.pr = nn.Linear(d, d)
        nn.init.zeros_(s.pr.weight); nn.init.zeros_(s.pr.bias)
    def forward(s, patch, etok):
        a, _ = s.at(s.n(patch), etok, etok, need_weights=False)
        return patch + s.pr(s.ln_a(a))

class Tier1Img(nn.Module):
    '''Ban r3: LayerScale `gamma` 1e-4 — chan cu sup ma zero-init khong chan duoc (0/16 epoch-arm
       o 57, truoc do sup 3 lan lien tiep).'''
    def __init__(s, d, nhead=8, drop=0.1, diff=True):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.nq = nn.LayerNorm(d); s.nk = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d)
        s.pr = nn.Linear(d, d); s.gamma = nn.Parameter(torch.full((d,), 1e-4)); s.diff = diff
    def forward(s, patch, prior, has):
        a, _ = s.at(s.nq(patch), s.nk(prior), s.nk(prior), need_weights=False)
        z = s.gamma * s.pr(s.ln_a(patch - a if s.diff else a))
        return patch + z * has.view(-1, 1, 1)

class MLDecoder(nn.Module):
    def __init__(s, C, d_in, d=512, nhead=8, ff=1024, drop=0.1, d_ehr_in=0, t1=False):
        super().__init__()
        s.q = nn.Embedding(C, d); s.q.weight.requires_grad_(False)   # query lop KHONG hoc (bai goc)
        s.pin = nn.Linear(d_in, d); s.img = XAttn(d, nhead, ff, drop)
        s.t1 = Tier1(d, nhead, drop) if t1 else None
        if t1: s.ein = nn.Linear(d_ehr_in, d)
        s.out = GroupFC(C, d)
    def forward(s, patch, etok=None):
        q = s.q.weight.unsqueeze(0).expand(patch.shape[0], -1, -1)
        kv = s.pin(patch)
        if s.t1 is not None: kv = s.t1(kv, s.ein(etok))
        h, _ = s.img(q, kv); return s.out(h)

class ASL(nn.Module):
    '''Verbatim nb11..59. gn=4, gp=0 la cau hinh 35 da chot.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lo = y*torch.log(xp.clamp(min=s.eps)) + (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y)
        return -(lo * torch.pow(1-pt, g)).mean()

from sklearn.metrics import average_precision_score
def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o
mac = lambda Y_, P_: float(np.nanmean(ap_cols(Y_, P_)))
tie = lambda Y_, P_: {t: float(np.nanmean(ap_cols(Y_, P_)[i])) for t, i in TIER.items()}
_r6 = np.random.default_rng(6); _y6 = (_r6.random((3000, C)) < 0.2).astype('float32')
_s6 = _r6.random((3000, C)).astype('float32')
assert np.nanmax(np.abs(ap_cols(_y6, _s6) - np.array(
    [average_precision_score(_y6[:, j], _s6[:, j]) for j in range(C)]))) < 1e-6
print('cong 1: ap_cols khop sklearn')

_m0 = timm.create_model(MODEL, pretrained=True, num_classes=0, in_chans=1)
_cfg = timm.data.resolve_model_data_config(_m0)
MU, SD = float(np.mean(_cfg['mean'])), float(np.mean(_cfg['std']))
_IN_SD = {k: v.clone() for k, v in _m0.state_dict().items()}; del _m0

def build_backbone():
    bb = timm.create_model(MODEL, pretrained=False, num_classes=0, in_chans=1)
    ck = torch.load(BB_USE, map_location='cpu', weights_only=False)
    miss = bb.load_state_dict({k: v for k, v in ck['sd'].items() if not k.startswith('head.')},
                              strict=False)
    bad = [k for k in miss.missing_keys if not k.startswith('head.')]
    assert not bad, f'thieu khoa khi nap backbone PadChest: {bad[:5]}'
    return bb, ck

_bb, _ck0 = build_backbone()
with torch.no_grad():
    _d = max((_IN_SD[k] - v).abs().max().item() for k, v in _bb.state_dict().items() if k in _IN_SD)
assert _d > 1e-4, (f'trong so GIONG HET ImageNet (lech {_d:.1e}) -> `convnextv2_padchest.pt` khong '
                   'duoc nap. Moi so ve do manh/corr se vo nghia.')
# cong 2: hinh dang feature map — KHONG tin gia dinh ve channels-first/last
_bb = _bb.to(dev).eval()
with torch.no_grad():
    _fm = _bb.forward_features(torch.zeros(2, 1, SIZE, SIZE, device=dev))
_shape = tuple(_fm.shape)
_CHLAST = _shape[-1] != _shape[-2]                # [B,H,W,Cf] neu chieu cuoi khac chieu ke cuoi
D_CXR = _shape[-1] if _CHLAST else _shape[1]
N_TOK = (_shape[1]*_shape[2]) if _CHLAST else (_shape[2]*_shape[3])
print(f'cong 2: feature map {_shape} -> {"channels-LAST" if _CHLAST else "channels-FIRST"} | '
      f'{N_TOK} token, d={D_CXR}')
assert 100 <= N_TOK <= 2000 and 256 <= D_CXR <= 2048, f'hinh dang bat thuong: {_shape}'
def to_tokens(fm):
    '''[B,H,W,Cf] hoac [B,Cf,H,W] -> [B, N, Cf]. Quyet dinh bang HINH DANG do duoc o cong 2.'''
    return fm.flatten(1, 2) if _CHLAST else fm.flatten(2).transpose(1, 2)
print(f'cong 3: trong so PadChest da nap | lech ImageNet {_d:.3f} | '
      f'ep {_ck0.get("ep", "?")} mAP PadChest {_ck0.get("mAP", float("nan")):.4f}')
del _bb; torch.cuda.empty_cache()

## C3 · Ba arm + cổng khoá trước

In [ ]:
ARMS = {
  'cnn_full': dict(img=True, ehr=True),   # arm thang bac 2 — DOC LAP, backbone cua rieng no
}
def build_arms(seed=SEED):
    bb, _ = build_backbone(); bb = bb.to(dev)
    M, E, I = nn.ModuleDict(), nn.ModuleDict(), nn.ModuleDict()
    for tag, cfg in ARMS.items():
        torch.manual_seed(seed)                  # gieo lai TRUOC TUNG arm -> khoi tao giong nhau
        M[tag] = MLDecoder(C, D_CXR, D_MLD, d_ehr_in=D_ETOK, t1=cfg['ehr'])
        if cfg['ehr']: E[tag] = ColTokBag(FEAT.shape[1], D_ETOK, V=V_MIC)
        if cfg['img']: I[tag] = Tier1Img(D_CXR)
    return bb, M.to(dev), E.to(dev), I.to(dev)

BB, MOD, ETK, IMG = build_arms()
NP_ = {t: sum(p.numel() for m in ([MOD[t]] + [x[t] for x in (ETK, IMG) if t in x])
              for p in m.parameters() if p.requires_grad) for t in ARMS}
print(f'{"arm":<14}{"tham so dau doc":>16}')
for t in ARMS: print(f'  {t:<12}{NP_[t]:>16,}')
print(f'  backbone dung chung: {sum(p.numel() for p in BB.parameters()):,} (CO huan luyen)')

_p = torch.randn(4, N_TOK, D_CXR, device=dev); _pp = torch.randn(4, N_TOK, D_CXR, device=dev)
_h1 = torch.ones(4, device=dev); _h0 = torch.zeros(4, device=dev)
_e = torch.randn(4, FEAT.shape[1], device=dev)
_mi = torch.zeros(4, MIC_L, dtype=torch.long, device=dev); _mc = torch.zeros(4, MIC_L, device=dev)
assert not next(iter(MOD.values())).q.weight.requires_grad, 'query_embed dang hoc'
with torch.no_grad():
    for t in IMG:
        _rel = float((IMG[t](_p, _pp, _h1) - _p).norm() / (_p.norm() + 1e-9))
        assert _rel < 1e-3, f'{t}: Tier1Img nhieu {_rel:.2e} luc khoi tao, phai < 1e-3'
        assert (IMG[t](_p, _pp, _h0) - _p).abs().max() < 1e-6, f'{t}: has=0 KHONG cat duoc nhanh'
    m = MOD['cnn_full'].eval(); et = ETK['cnn_full'](_e, _mi, _mc)
    l1 = m(_p, et); _k = m.t1; m.t1 = None; l2 = m(_p, et); m.t1 = _k
    assert (l1 - l2).abs().max() < 1e-5, 'Tier1 KHONG zero-init'
print('cong 4: Tier1 zero-init | Tier1Img LayerScale + has=0 cat sach OK')
BB, MOD, ETK, IMG = build_arms()

## C4 · Huấn luyện — một forward backbone cho **cả ba** arm

Backbone tính **một lần** mỗi batch (hai lần nếu có arm dùng phim trước) rồi ba arm dùng chung.
`56` mắc lỗi gọi lại forward cho từng arm và chạy 15 img/s; ở đây tránh hẳn.

Backbone **có huấn luyện** (khác `53`/`57` nơi RAD-DINO đông cứng) — ConvNeXt rẻ hơn 11 lần nên
khả thi, và `57` đã cho thấy backbone đông cứng không thích nghi thêm được gì.

In [ ]:
# ==== RESET s42: xoa co done + xoa file cu tren Drive, de C4 huan luyen lai tu dau ====
# Xoa file cu vi ghi de cung ten qua Drive FUSE chi giu lan ghi DAU -> trong so moi se
# khong xuong dia neu de w_s42_* cu nam do.
import glob as _g
SD = 42
for d in {RES_LOC, RES41D}:
    for f in _g.glob(os.path.join(d, f'*_s{SD}_ep*.npy')) + _g.glob(os.path.join(d, f'w_s{SD}_ep*.pt')) \
             + _g.glob(os.path.join(d, f'cnn_arch_s{SD}.pt')):
        os.remove(f); print('xoa', f.replace(OUT_DIR, '...'))
_lg = find('epoch_log.json'); assert _lg
LOG = json.load(open(_lg[0]))
LOG.pop(f's{SD}', None)
put('epoch_log.json', lambda q: json.dump(LOG, open(q, 'w')))
print(f'nhat ky con: {list(LOG)} -> C4 se huan luyen lai s{SD}')


In [ ]:
from torchvision import transforms as T2
crit = ASL(gn=4, gp=0).to(dev)
sig = lambda x: 1.0 / (1.0 + np.exp(-x))
MIDS_T, MCNT_T = torch.from_numpy(MIDS), torch.from_numpy(MCNT)
FEAT_T = torch.from_numpy(FEAT)
tf_tr = T2.Compose([T2.RandomResizedCrop(SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05),
                                         interpolation=T2.InterpolationMode.BICUBIC),
                    T2.RandomRotation(7)])
NEED_PREV = any(c['img'] for c in ARMS.values())

class DS(Dataset):
    def __init__(s, idx, train): s.idx, s.train = np.asarray(idx), train
    def __len__(s): return len(s.idx)
    def _img(s, i):
        im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        if s.train: im = tf_tr(im)
        x = torch.from_numpy(np.asarray(im, dtype='uint8').copy()).unsqueeze(0).float().div_(255)
        return (x - MU) / SD
    def __getitem__(s, k):
        i = int(s.idx[k]); pr = int(PREV_ROW[i]); x = s._img(i)
        return dict(image=x, image_prev=(s._img(pr) if (NEED_PREV and pr >= 0)
                                         else torch.zeros_like(x)),
                    has=torch.tensor(1.0 if pr >= 0 else 0.0), row=i, pos=k,
                    y=torch.from_numpy(YM[i]))

def mk_loader(idx, train, bs, nw=NW):
    return DataLoader(DS(idx, train), batch_size=bs, shuffle=train, num_workers=nw,
                      pin_memory=True, drop_last=train, persistent_workers=nw > 0,
                      prefetch_factor=PREFETCH if nw > 0 else None)
_DLC = {}
def eval_loader(idx, bs):
    '''Tao MOT LAN roi dung lai — tao moi moi epoch se ngap spam teardown va ton thoi gian.'''
    k = (len(idx), int(idx[0]), int(idx[-1]), bs)
    if k not in _DLC: _DLC[k] = mk_loader(idx, False, bs)
    return _DLC[k]

def arm_forward(t, p, pp, row, has):
    cfg = ARMS[t]
    q = IMG[t](p, pp, has) if cfg['img'] else p
    et = (ETK[t](FEAT_T[row].to(dev), MIDS_T[row].to(dev), MCNT_T[row].to(dev))
          if cfg['ehr'] else None)
    return MOD[t](q, et)

@torch.no_grad()
def predict(idx, bs=None):
    bs = BS_EVAL if bs is None else bs      # 128 o 518 se TRAN RAM o 1022
    BB.eval(); MOD.eval(); ETK.eval(); IMG.eval()
    P = {t: np.zeros((len(idx), C), 'float32') for t in ARMS}
    for b in tqdm(eval_loader(idx, bs), desc='eval', leave=False):
        with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
            p = to_tokens(BB.forward_features(b['image'].to(dev, non_blocking=True))).float()
            pp = (to_tokens(BB.forward_features(b['image_prev'].to(dev, non_blocking=True))).float()
                  if NEED_PREV else p)
        hs = b['has'].to(dev)
        for t in ARMS:
            P[t][b['pos'].numpy()] = torch.sigmoid(
                arm_forward(t, p, pp, b['row'], hs).float()).cpu().numpy()
    assert not any(np.isnan(v).any() for v in P.values()), 'predict cho ra NaN'
    return P

def _tim_log():
    """Do CA thu muc trung ten (Drive de ra 'stage4_nbXX (1)' / 'r1022_E8 (1)').
       Thieu buoc nay thi LOG = {} va moi seed bi huan luyen LAI tu dau."""
    out = list(find('epoch_log.json'))
    _goc, _run = os.path.basename(os.path.dirname(RES41D)), os.path.basename(RES41D)
    for x in sorted(os.listdir(OUT_DIR)):
        if not (x == _goc or x.startswith(_goc + ' (')): continue
        _b = os.path.join(OUT_DIR, x)
        for y in sorted(os.listdir(_b)):
            if not (y == _run or y.startswith(_run + ' (')): continue
            p = os.path.join(_b, y, 'epoch_log.json')
            if os.path.exists(p) and p not in out: out.append(p)
    # KHONG sap theo KICH THUOC: file to nhat chua chac day du nhat (co the nhieu epoch cua
    # mot seed nhung THIEU co `done`). Cham theo NOI DUNG: so seed da `done`, roi so muc epoch.
    def _diem(p):
        try: d = json.load(open(p))
        except Exception: return (-1, -1)
        return (sum(1 for v in d.values() if isinstance(v, dict) and v.get('done')),
                sum(len([k for k in v if k.isdigit()]) for v in d.values() if isinstance(v, dict)))
    out = sorted(out, key=lambda p: (-_diem(p)[0], -_diem(p)[1]))
    for p in out[:4]:
        _s, _e = _diem(p)
        print(f'   ung vien: seed done={_s} epoch={_e}  {p}')
    return out


def _tim(pat):
    """Nhu `find` nhung do CA thu muc trung ten. Dung cho moi phep kiem file tren Drive."""
    out = list(find(pat))
    _goc, _run = os.path.basename(os.path.dirname(RES41D)), os.path.basename(RES41D)
    for x in sorted(os.listdir(OUT_DIR)):
        if not (x == _goc or x.startswith(_goc + ' (')): continue
        _b = os.path.join(OUT_DIR, x)
        for y in sorted(os.listdir(_b)):
            if not (y == _run or y.startswith(_run + ' (')): continue
            p = os.path.join(_b, y, pat)
            if os.path.exists(p) and p not in out: out.append(p)
    return out


_old = _tim_log()
print(f'nhat ky tim thay: {len(_old)} ban' + (f' -> {_old[0]}' if _old else ' (se huan luyen tu dau)'))
def _gop_log(paths):
    """Nhat ky bi PHAN MANH qua cac thu muc trung ten: moi session ghi mot ban rieng, KHONG
       ban nao co du ca hai seed (vd ban tot nhat chi co s43). Chon bat cu ban le nao cung
       thieu -> seed vang bi huan luyen LAI. Gop theo TUNG SEED, giu ban tot nhat cua seed do."""
    G = {}
    for p in paths:
        try: d = json.load(open(p))
        except Exception: continue
        for sk, v in d.items():
            if not isinstance(v, dict): continue
            diem = (1 if v.get('done') else 0, len([k for k in v if k.isdigit()]))
            if sk not in G or diem > G[sk][1]: G[sk] = (v, diem)
    return {sk: v for sk, (v, _) in G.items()}


LOG = _gop_log(_old)
print('nhat ky sau khi GOP: ' + (', '.join(
    f'{sk}: {len([k for k in v if k.isdigit()])} epoch, done={bool(v.get("done"))}'
    for sk, v in sorted(LOG.items())) or '(rong)'))
# Doi tap arm roi resume len nhat ky cu -> bang ket qua tron hai cau hinh ma khong bao gi.
for _sk, _v in LOG.items():
    _eps = [k for k in _v if k.isdigit()]
    if _eps:
        _old_arms = list(_v[_eps[0]])
        assert set(_old_arms) == set(ARMS), (
            f'nhat ky co {len(_old_arms)} arm {_old_arms}, notebook co {len(ARMS)} '
            f'{list(ARMS)}. DOI RUN_TAG o C0 (vd r2) roi chay lai. KHONG xoa gi.')
# Dat `CHI_HAM = True` o mot cell TRUOC (hoac sua o day) de chi DINH NGHIA ham roi dung —
# dung khi chi can chay cell suy luan TEST phia sau, khong muon huan luyen lai.
if globals().get('CHI_HAM', False):
    print(f'CHI_HAM = True -> da dinh nghia predict/arm_forward/eval_loader, KHONG huan luyen. '
          f'Nhat ky: {list(LOG)}')
    raise SystemExit

for seed in SEEDS:
    sk = f's{seed}'
    # NGUON SU THAT LA FILE, khong phai nhat ky.
    # 2026-09-13: nhat ky bi phan manh qua 9 thu muc trung ten -> `s42` chi con 7 muc va
    # KHONG co co `done`, trong khi luc gom thu muc dem duoc DU 8/8/8 file. Tin nhat ky thi
    # huan luyen lai 23 gio cho mot seed da chay xong. Tin FILE thi khong bao gio nham.
    _thieu = [f'{p}_cnn_full_{sk}_ep{e}.npy' for e in range(1, EPOCHS + 1)
              for p in ('p', 'pte') if not _tim(f'{p}_cnn_full_{sk}_ep{e}.npy')]
    _w_thieu = [f'w_{sk}_ep{e}.pt' for e in range(1, EPOCHS + 1) if not _tim(f'w_{sk}_ep{e}.pt')]
    print(f'{sk}: du doan {2*EPOCHS - len(_thieu)}/{2*EPOCHS}, trong so '
          f'{EPOCHS - len(_w_thieu)}/{EPOCHS}, nhat ky done={bool(LOG.get(sk, {}).get("done"))}')
    if not _thieu:
        print(f'   -> DU FILE, bo qua (khong can co `done` trong nhat ky)'); continue
    if LOG.get(sk, {}).get('done') or not _w_thieu:
        raise AssertionError(
            f'{sk}: THIEU {len(_thieu)}/{2*EPOCHS} file du doan (vd {_thieu[0]}) nhung trong so '
            f'{"con du" if not _w_thieu else "cung thieu"}. Trong so con -> TAI SINH bang suy '
            f'luan (~20 phut/epoch), RE HON huan luyen lai {EPOCHS} epoch. Muon huan luyen lai '
            f'that thi xoa trong so cua {sk} roi chay lai.')
    BB, MOD, ETK, IMG = build_arms(seed)
    PARAM = [p for m in (MOD, ETK, IMG) for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(
        [{'params': [p for p in PARAM if p.ndim > 1], 'weight_decay': WD, 'lr': LR*10},
         {'params': [p for p in PARAM if p.ndim <= 1], 'weight_decay': 0.0, 'lr': LR*10},
         {'params': [p for p in BB.parameters() if p.ndim > 1], 'weight_decay': WD, 'lr': LR},
         {'params': [p for p in BB.parameters() if p.ndim <= 1], 'weight_decay': 0.0, 'lr': LR}])
    steps = math.ceil(len(I_TF) / (BATCH*ACCUM)) * EPOCHS; warm = int(WARMUP*steps)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s/max(warm,1) if s < warm else
                                            0.5*(1+math.cos(math.pi*(s-warm)/max(steps-warm,1))))
    dl = mk_loader(I_TF, True, BATCH)
    LOG.setdefault(sk, {})
    for ep in range(EPOCHS):
        BB.train(); MOD.train(); ETK.train(); IMG.train()
        t0, tot, nb = time.time(), {t: 0.0 for t in ARMS}, 0
        opt.zero_grad(set_to_none=True)
        for j, b in enumerate(tqdm(dl, desc=f'{sk} ep{ep+1}/{EPOCHS}', leave=False)):
            with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
                p = to_tokens(BB.forward_features(b['image'].to(dev, non_blocking=True))).float()
                pp = (to_tokens(BB.forward_features(
                          b['image_prev'].to(dev, non_blocking=True))).float()
                      if NEED_PREV else p)
            y = b['y'].to(dev); hs = b['has'].to(dev); loss = 0.0
            for t in ARMS:
                lg = arm_forward(t, p, pp, b['row'], hs)
                if not torch.isfinite(lg).all(): continue
                l_ = crit(lg.float(), y); loss = loss + l_; tot[t] += float(l_.detach())
            if not torch.is_tensor(loss): opt.zero_grad(set_to_none=True); continue
            (loss / ACCUM).backward(); nb += 1
            if (j + 1) % ACCUM == 0:
                gn = torch.nn.utils.clip_grad_norm_(PARAM + list(BB.parameters()), CLIP,
                                                    error_if_nonfinite=False)
                if not torch.isfinite(gn):       # total_norm = nan se BOI BAN toan bo tham so
                    opt.zero_grad(set_to_none=True); continue
                opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
        PV = predict(I_VA)
        for t in ARMS:
            put(f'p_{t}_{sk}_ep{ep+1}.npy', lambda q, v=PV[t]: np.save(q, v.astype('float16')))
        # KHONG luu `psl_`: lat chon da NAM TRONG train, du doan tren no la TRONG MAU.
        # ANH CHUP TRONG SO tung epoch, ten RIENG -> khong bao gio bi ghi de. `53/r3` mat sach
        # trong so ep2-10 vi ghi de MOT file qua FUSE va cac lan ghi sau ep1 hong IM LANG.
        put(f'w_{sk}_ep{ep+1}.pt', lambda q: torch.save(
            dict(bb=BB.state_dict(), mod=MOD.state_dict(), etk=ETK.state_dict(),
                 img=IMG.state_dict(), seed=seed, ep=ep, run=RUN_TAG), q))
        if DO_TEST:                     # MOI epoch — bo luat last-3, de do test o do sau THAT
            PT = predict(I_TE)                            # tinh XONG roi moi dung toi file
            for t in ARMS:
                put(f'pte_{t}_{sk}_ep{ep+1}.npy', lambda q, v=PT[t]: np.save(q, v.astype('float16')))
            try:
                with open(P_TOUCH, 'a') as _f:
                    _f.write(json.dumps(dict(nb='64b', run=RUN_TAG, seed=seed, ep=ep+1,
                                             n=len(I_TE), muc_dich='luot chot R2a',
                                             t=datetime.datetime.now().isoformat(timespec='seconds')
                                             )) + chr(10))
            except Exception as _e:
                print(f'  !! ghi _test_touches that bai ({type(_e).__name__}) — mount?')
        # Tier1Img da lam SUP arm 3 lan (53 imgz, 56 tpb, 57 r2 tpb). LayerScale chan duoc o
        # 57 (0/16 epoch-arm) nhung van phai THEO DOI: `ratio_img` = ‖phan du‖/‖patch‖ la dai
        # luong vuot ~1 thi sup; `pr_norm` chi la so gian tiep.
        with torch.no_grad():
            _bd = next(iter(eval_loader(I_VA[:BS_DIAG], BS_DIAG))); _ri = {}
            with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
                _pb = to_tokens(BB.forward_features(_bd['image'].to(dev))).float()
                _ppb = to_tokens(BB.forward_features(_bd['image_prev'].to(dev))).float()
            _hs2 = _bd['has'].to(dev)
            for t in IMG:
                _ri[t] = float(((IMG[t](_pb, _ppb, _hs2) - _pb).norm(dim=-1)
                                / (_pb.norm(dim=-1) + 1e-9)).mean())
        LOG[sk][str(ep)] = {t: dict(mAP_anh=mac(YM[VA], PV[t]),   # KHONG co `sel`: xem tren
                                    pr_norm=(float(IMG[t].pr.weight.norm()) if t in IMG else 0.0),
                                    ratio_img=_ri.get(t, 0.0), **tie(YM[VA], PV[t]))
                            for t in ARMS}
        put('epoch_log.json', lambda q: json.dump(LOG, open(q, 'w')))
        drive_gate(f'{sk} ep{ep+1}')             # hong thi DUNG NGAY, khong chay tiep 10 gio
        _vr = torch.cuda.max_memory_allocated() / 1e9
        _tot = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'   VRAM dinh diem {_vr:.1f} / {_tot:.0f} GB ({_vr/_tot:.0%})'
              + ('   -> con du: co the nang BATCH va ha ACCUM, GIU EFF = 288'
                 if _vr / _tot < 0.45 else ''))
        torch.cuda.reset_peak_memory_stats()
        print(f'{sk} ep{ep+1:>2}/{EPOCHS} | {nb*BATCH/max(time.time()-t0,1e-9):.0f} img/s | '
              f'{(time.time()-t0)/60:.1f} ph')
        print('   VAL      ' + ' '.join(f'{t}:{mac(YM[VA], PV[t]):.4f}' for t in ARMS))
        print('   ^ VAL chi de THEO DOI — E=8 da KHOA, khong duoc doi gi theo dong nay')
        print('   TAIL     ' + ' '.join(f'{t}:{tie(YM[VA], PV[t])["tail"]:.4f}' for t in ARMS))
        if _ri:
            print('   ratio_img ' + ' '.join(f'{t}:{v:.4f}' for t, v in _ri.items())
                  + '   (vuot ~1 = phan du AT patch -> sup)')
    put(f'cnn_arch_{sk}.pt', lambda q: torch.save(
        dict(bb=BB.state_dict(), mod=MOD.state_dict(), etk=ETK.state_dict(),
             img=IMG.state_dict(), seed=seed), q))
    LOG[sk]['done'] = True; put('epoch_log.json', lambda q: json.dump(LOG, open(q, 'w')))
print('huan luyen xong')

## C4b · Chỉ **định nghĩa hàm** — bản rút gọn của C4, KHÔNG huấn luyện

Cắt nguyên văn phần đầu của C4: `DS`, `mk_loader`, `eval_loader`, `arm_forward`,
`predict`. Bỏ toàn bộ vòng huấn luyện và phần đọc nhật ký.

Dùng khi chỉ cần chạy cell suy luận TEST. **Chạy cell này thay cho C4.**


In [ ]:
from torchvision import transforms as T2
crit = ASL(gn=4, gp=0).to(dev)
sig = lambda x: 1.0 / (1.0 + np.exp(-x))
MIDS_T, MCNT_T = torch.from_numpy(MIDS), torch.from_numpy(MCNT)
FEAT_T = torch.from_numpy(FEAT)
tf_tr = T2.Compose([T2.RandomResizedCrop(SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05),
                                         interpolation=T2.InterpolationMode.BICUBIC),
                    T2.RandomRotation(7)])
NEED_PREV = any(c['img'] for c in ARMS.values())

class DS(Dataset):
    def __init__(s, idx, train): s.idx, s.train = np.asarray(idx), train
    def __len__(s): return len(s.idx)
    def _img(s, i):
        im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        if s.train: im = tf_tr(im)
        x = torch.from_numpy(np.asarray(im, dtype='uint8').copy()).unsqueeze(0).float().div_(255)
        return (x - MU) / SD
    def __getitem__(s, k):
        i = int(s.idx[k]); pr = int(PREV_ROW[i]); x = s._img(i)
        return dict(image=x, image_prev=(s._img(pr) if (NEED_PREV and pr >= 0)
                                         else torch.zeros_like(x)),
                    has=torch.tensor(1.0 if pr >= 0 else 0.0), row=i, pos=k,
                    y=torch.from_numpy(YM[i]))

def mk_loader(idx, train, bs, nw=NW):
    return DataLoader(DS(idx, train), batch_size=bs, shuffle=train, num_workers=nw,
                      pin_memory=True, drop_last=train, persistent_workers=nw > 0,
                      prefetch_factor=PREFETCH if nw > 0 else None)
_DLC = {}
def eval_loader(idx, bs):
    '''Tao MOT LAN roi dung lai — tao moi moi epoch se ngap spam teardown va ton thoi gian.'''
    k = (len(idx), int(idx[0]), int(idx[-1]), bs)
    if k not in _DLC: _DLC[k] = mk_loader(idx, False, bs)
    return _DLC[k]

def arm_forward(t, p, pp, row, has):
    cfg = ARMS[t]
    q = IMG[t](p, pp, has) if cfg['img'] else p
    et = (ETK[t](FEAT_T[row].to(dev), MIDS_T[row].to(dev), MCNT_T[row].to(dev))
          if cfg['ehr'] else None)
    return MOD[t](q, et)

@torch.no_grad()
def predict(idx, bs=None):
    bs = BS_EVAL if bs is None else bs      # 128 o 518 se TRAN RAM o 1022
    BB.eval(); MOD.eval(); ETK.eval(); IMG.eval()
    P = {t: np.zeros((len(idx), C), 'float32') for t in ARMS}
    for b in tqdm(eval_loader(idx, bs), desc='eval', leave=False):
        with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
            p = to_tokens(BB.forward_features(b['image'].to(dev, non_blocking=True))).float()
            pp = (to_tokens(BB.forward_features(b['image_prev'].to(dev, non_blocking=True))).float()
                  if NEED_PREV else p)
        hs = b['has'].to(dev)
        for t in ARMS:
            P[t][b['pos'].numpy()] = torch.sigmoid(
                arm_forward(t, p, pp, b['row'], hs).float()).cpu().numpy()
    assert not any(np.isnan(v).any() for v in P.values()), 'predict cho ra NaN'
    return P

print('C4b: da dinh nghia ' + ', '.join(['DS', 'mk_loader', 'eval_loader', 'arm_forward', 'predict']) + ' — KHONG huan luyen. Chay tiep cell suy luan TEST.')


## C+ · TEST với **XGB mới** — suy luận lại `pte_`, KHÔNG huấn luyện

`52a` điền hằng số cho test ở 40 cột XGB; `nb69` C7 **đã** huấn luyện lại 40 cây trên lát
khớp và ghi `xgb_oof_co_test.npy`. Cell này nạp file đó, dựng lại `FEAT_T`, rồi chạy
`predict(I_TE)` với trọng số `w_` từng epoch đã lưu — **không huấn luyện lại gì**.

Ghi vào thư mục **mới** `<RUN_TAG>_xgbnew`: không đè `pte_` cũ, vì ghi đè cùng tên qua
Drive FUSE chỉ giữ lần ghi đầu. Dự đoán **val không cần chạy lại** — hàng val của `FEAT`
giữ nguyên từng bit. Cell **không đọc nhãn test**, không tốn lần chạm nào.

Chạy sau cell huấn luyện (cell đó sẽ in "đã xong, bỏ qua" chứ không huấn luyện).


In [ ]:
# ==== TEST voi XGB MOI — suy luan lai pte_ tu w_ tung epoch, KHONG huan luyen ====
# Chay SAU cell huan luyen (cell dinh nghia `predict`). KHONG doc nhan test.

# --- tim xgb_oof_co_test.npy, chiu duoc thu muc TRUNG TEN cua Drive ---
_cand = []
for _x in sorted(os.listdir(OUT_DIR)):
    if _x == 'stage4_nb69' or _x.startswith('stage4_nb69 ('):
        _p = os.path.join(OUT_DIR, _x, 'xgb_oof_co_test.npy')
        if os.path.exists(_p): _cand.append(_p)
assert _cand, ('khong thay xgb_oof_co_test.npy trong stage4_nb69*. nb69 C7 DA chay va ghi file '
               'nay; neu khong thay thi kiem thu muc trung ten tren Drive truoc khi chay lai C7.')
P_OOF_NEW = max(_cand, key=os.path.getsize)
print(f'XGB moi: {P_OOF_NEW.replace(OUT_DIR, "...")}')

_XGF2 = np.load(P_OOF_NEW).astype('float32')
_XGF1 = np.load(P_OOF).astype('float32')
assert _XGF2.shape == (len(ORDER), C) and np.isfinite(_XGF2).all()
assert np.array_equal(_XGF2[~TE], _XGF1[~TE]), 'file moi doi ca hang train/val -> KHONG dung'
assert not np.allclose(_XGF2[TE], _XGF1[TE]), 'hang test khong doi -> file khong phai ban moi'
_cl = lambda a: np.clip(a, 1e-6, 1 - 1e-6)
_l2 = np.log(_cl(_XGF2) / (1 - _cl(_XGF2)))
XGB_F2 = np.clip((_l2 - _l2[TF_].mean(0, keepdims=True)) / (_l2[TF_].std(0, keepdims=True) + 1e-6),
                 -EHR_CLIP, EHR_CLIP).astype('float32')
FEAT2 = np.concatenate([EHR_COL, XGB_F2], 1)
assert np.array_equal(FEAT2[~TE], FEAT[~TE]), 'hang train/val phai giu nguyen tung bit'
assert not np.allclose(FEAT2[TE], FEAT[TE]), 'hang test phai DOI'
FEAT_T = torch.from_numpy(FEAT2)                 # `arm_forward` doc bien TOAN CUC nay
print(f'FEAT_T <- XGB moi | do lech chuan 40 cot o test: {XGB_F2[TE].std(0).mean():.4f} '
      f'(ban cu {XGB_F[TE].std(0).mean():.4f}, = 0 vi la hang so)')

OUT2 = RES41D + '_xgbnew'


def _mkdir_chac(p, thu=5):
    """`os.path.isdir` tren Drive FUSE tra True cho thu muc CHUA ton tai, va `makedirs`
       co the im lang khong tao duoc -> phai GHI THAT roi DOC LAI moi biet."""
    for i in range(thu):
        try: os.makedirs(p, exist_ok=True)
        except Exception: pass
        t = os.path.join(p, '_probe.txt')
        try:
            with open(t, 'w') as _f: _f.write('x' * 64)
            with open(t) as _f: _ok = _f.read() == 'x' * 64
            os.remove(t)
            if _ok: return True
        except Exception: pass
        time.sleep(2 * (i + 1))
    return False


assert _mkdir_chac(OUT2), f'khong tao duoc {OUT2} tren Drive -> mount hong'


def put2(name, arr=None):
    """arr=None: file local DA co (lan chay truoc) -> chi day len Drive."""
    lp = os.path.join(RES_LOC, 'xgbnew_' + name)
    if arr is not None: np.save(lp, arr.astype('float16'))
    assert os.path.exists(lp), f'thieu ban local {lp}'
    dp = os.path.join(OUT2, name)
    for i in range(4):
        try:
            shutil.copy(lp, dp)
            if os.path.getsize(dp) == os.path.getsize(lp): return
        except Exception:
            _mkdir_chac(OUT2)                      # thu muc bien mat -> tao lai roi thu tiep
        time.sleep(2 * (i + 1))
    raise OSError(f'khong ghi duoc {name} len Drive sau 4 lan')


# --- day nhung file da suy luan o lan chay truoc (con o /content) len Drive ---
_day = 0
for _sd in SEEDS:
    for _ep in range(1, EPOCHS + 1):
        for _t in ARMS:
            _nm = f'pte_{_t}_s{_sd}_ep{_ep}.npy'
            if os.path.exists(os.path.join(RES_LOC, 'xgbnew_' + _nm)) \
                    and not os.path.exists(os.path.join(OUT2, _nm)):
                put2(_nm); _day += 1
if _day: print(f'day {_day} file da co san o local len Drive (khong phai suy luan lai)')


JOBS = [(sd, ep) for sd in SEEDS for ep in range(1, EPOCHS + 1)
        if not os.path.exists(os.path.join(OUT2, f'pte_cnn_full_s{sd}_ep{ep}.npy'))]
print(f'can suy luan {len(JOBS)}/{len(SEEDS) * EPOCHS} checkpoint '
      f'-> {OUT2.replace(OUT_DIR, "...")}')
_cur, _t0 = None, time.time()
for k, (sd, ep) in enumerate(JOBS, 1):
    if _cur != sd:
        BB, MOD, ETK, IMG = build_arms(sd); _cur = sd
    _w = find(f'w_s{sd}_ep{ep}.pt')
    assert _w, f'thieu w_s{sd}_ep{ep}.pt -> kiem thu muc trung ten tren Drive'
    w = torch.load(max(_w, key=os.path.getsize), map_location='cpu', weights_only=False)
    assert int(w['seed']) == sd and int(w['ep']) == ep - 1, f'w_s{sd}_ep{ep}.pt ghi seed/ep khac ten'
    BB.load_state_dict(w['bb']); MOD.load_state_dict(w['mod'])
    ETK.load_state_dict(w['etk']); IMG.load_state_dict(w['img']); del w
    PT = predict(I_TE)
    for t in ARMS: put2(f'pte_{t}_s{sd}_ep{ep}.npy', PT[t])
    print(f'  [{k}/{len(JOBS)}] s{sd} ep{ep}  {(time.time() - _t0) / 60:5.1f} ph', flush=True)

_n = sum(os.path.exists(os.path.join(OUT2, f'pte_cnn_full_s{sd}_ep{ep}.npy'))
         for sd in SEEDS for ep in range(1, EPOCHS + 1))
print(f'\nxong {(time.time() - _t0) / 60:.1f} ph | {_n}/{len(SEEDS) * EPOCHS} file pte_ trong '
      f'{OUT2.replace(OUT_DIR, "...")}')
assert _n == len(SEEDS) * EPOCHS, 'chua du file -> chay lai cell nay (co resume)'
print('CHUA cham diem test. Buoc sau: nb61 C14.')


## C5 · Chọn arm trên lát chọn → blend → cổng

In [3]:
# ==== BAO CAO: 68b @1022 vs 64b @518, CUNG do sau epoch va CUNG thang tier ====
# C5 chay duoc chi voi C0 — KHONG can C1-C4. Quan trong khi Colab dut giua chung:
# C1 bung lai 377k anh (~30 phut) ma cell nay khong dung tam nao.
import glob as _glob

# `RES_LOC` la /content, mat theo session. Ban tren Drive van con vi `put()` o C4 da
# sao len va xac minh kich thuoc. `find()` (C0) quet CA HAI noi.
_lg = find('epoch_log.json')
assert _lg, (f'khong thay epoch_log.json trong {SEARCH}. Neu C4 chua chay xong seed nao '
             'thi chua co gi de bao cao.')
LOG = json.load(open(_lg[0]))
print(f'nhat ky doc tu: {_lg[0]}')

# `ap_cols` von o C2 va `ARMS` o C3; dinh nghia lai ngay day de khoi phai chay hai cell
# do (C3 con keo theo C1). Ban sao VERBATIM — sua mot ben thi phai sua ben kia.
ARMS = {'cnn_full': dict(img=True, ehr=True)}


def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o


EPk = str(EPOCHS - 1)
R = pd.DataFrame([dict(seed=s, arm=t, **v) for s in SEEDS if EPk in LOG.get(f's{s}', {})
                  for t, v in LOG[f's{s}'][EPk].items()])
assert len(R), 'chua co seed nao chay xong'
SD = [s for s in SEEDS if EPk in LOG.get(f's{s}', {})]
print(f'=== 68b @ {SIZE}px — epoch cuoi (ep{EPOCHS}), {len(SD)} seed ===')
print(R.pivot_table(index='arm', values=['mAP_anh', 'head', 'medium', 'tail'], aggfunc='mean')
       .reindex(list(ARMS)).to_string(float_format=lambda v: f'{v:.4f}'))

def _ens(goc, run, arm, eps_toi, seeds):
    base, nm, uq = os.path.dirname(os.path.join(OUT_DIR, goc)), os.path.basename(goc), {}
    for d in os.listdir(base):
        if not (d == nm or d.startswith(nm + ' (')): continue
        p = os.path.join(base, d, run)
        if not os.path.isdir(p): continue
        for f in sorted(_glob.glob(os.path.join(p, f'p_{arm}_s*_ep*.npy'))):
            m = re.search(r'_s(\d+)_ep(\d+)\.npy$', os.path.basename(f))
            if m and int(m.group(2)) <= eps_toi and int(m.group(1)) in seeds:
                uq.setdefault(os.path.basename(f), f)
    if not uq: return None, 0
    Q = np.mean([np.load(f).astype('float32') for f in uq.values()], 0)
    assert Q.shape == (int(VA.sum()), C), f'{goc}: hinh dang {Q.shape}'
    return Q, len(uq)

PHI, nHI = _ens('stage4_nb68b', RUN_TAG, 'cnn_full', EPOCHS, set(SD))
PLO, nLO = _ens('stage4_nb64b', 'r2a',   'cnn_full', EPOCHS, set(SD))
print(f'\nensemble CUNG do sau (ep<={EPOCHS}, seed {SD}): '
      f'68b@{SIZE} {nHI} file | 64b@518 {nLO} file')
assert PHI is not None and PLO is not None, 'thieu mot ben'
# 2026-09-13: luot E=8 in bang 8 vs 16 file (s42 khong duoc glob liet ke) va bao TRUOT —
# so lech co ensemble thi khong doc duoc gi. Phai CHAN, khong duoc in.
assert nHI == nLO, (f'LECH SO FILE: 68b {nHI} vs 64b {nLO} -> phep so tron lan co ensemble. '
                    'Do theo TEN (DriveFS hay khong liet ke duoc) hoac chay lai seed thieu.')
aHI, aLO = ap_cols(YM[VA], PHI), ap_cols(YM[VA], PLO)

# doi chieu: ViT o cung phep thu (nb68) da tut bao nhieu
VIT = {'macro': -0.0055, 'head': -0.0048, 'medium': +0.0029, 'tail': -0.0138}
print(f'\n{"":<12}{"64b @518":>11}{"68b @"+str(SIZE):>12}{"chenh":>9}{"ViT (nb68)":>12}')
DL = {}
for nm_, cols in [('macro', np.arange(C))] + [(t, TIER[t]) for t in ('head', 'medium', 'tail')]:
    x, y = float(np.nanmean(aLO[cols])), float(np.nanmean(aHI[cols]))
    DL[nm_] = y - x
    print(f'  {nm_:<10}{x:>11.4f}{y:>12.4f}{y-x:>+9.4f}{VIT[nm_]:>+12.4f}')

BAY = ['Infiltration', 'Tortuous Aorta', 'Pleural Thickening', 'Adenopathy',
       'Calcification of the Aorta', 'Fissure', 'Granuloma']
C7 = np.array([LAB40.index(l) for l in BAY if l in LAB40])
C6 = np.array([j for j in C7 if LAB40[j] != 'Infiltration'])
print(f'\n=== 7 nhan medium ===')
print(f'{"nhan":<28}{"64b @518":>11}{"68b @"+str(SIZE):>12}{"chenh":>9}')
for j in C7[np.argsort(aLO[C7])]:
    print(f'  {LAB40[j]:<26}{aLO[j]:>11.4f}{aHI[j]:>12.4f}{aHI[j]-aLO[j]:>+9.4f}'
          + ('   <- nhan BAN, loai khoi cong' if LAB40[j] == 'Infiltration' else ''))
d7 = float(np.nanmean(aHI[C6]) - np.nanmean(aLO[C6]))
print(f'\n  6 nhan (bo Infiltration): {np.nanmean(aLO[C6]):.4f} -> {np.nanmean(aHI[C6]):.4f}'
      f'   {d7:+.4f}')

print(f'\n=== 10 nhan LOI / HAI nhat ===')
_o = np.argsort(aHI - aLO); _t = {j: t for t, c in TIER.items() for j in c}
for ten, idx in (('HAI nhat', _o[:10]), ('LOI nhat', _o[::-1][:10])):
    print(f'  --- {ten} ---')
    for j in idx:
        print(f'    {LAB40[j]:<26}{aLO[j]:>9.4f} -> {aHI[j]:<9.4f}{aHI[j]-aLO[j]:>+9.4f}'
              f'  {_t[j]:<7} prev {_prev[j]*100:6.3f}%')

print('\n' + '=' * 74)
print('CONG KHOA TRUOC (3 dieu kien, phai DAT CA BA)')
c1 = DL['tail'] >= -0.003
c2 = d7 >= 0.01
c3 = DL['macro'] >= -0.0007
print(f'  1. tail khong tut qua -0.003   : {DL["tail"]:+.4f}  {"DAT" if c1 else "TRUOT"}'
      f'   (ViT tut {VIT["tail"]:+.4f} — day la thu can bac bo)')
print(f'  2. 6 nhan medium tang >= +0.01 : {d7:+.4f}  {"DAT" if c2 else "TRUOT"}')
print(f'  3. macro >= 64b - 0.0007       : {DL["macro"]:+.4f}  {"DAT" if c3 else "TRUOT"}')
if c1 and c2 and c3:
    print('\n-> DAT CA BA. Gia thuyet "CNN tan dung do phan giai tot hon ViT" DUOC UNG HO.')
    print('   Buoc tiep: CHE_DO_THU=False (E=8, 2 seed) roi dua vao ensemble o 61/67.')
elif c1 and DL['macro'] > 0:
    print('\n-> tail KHONG bi hai (khac han ViT) va macro len, nhung 7 nhan medium chua du cong.')
    print('   Doc bang LOI/HAI de xem loi ich nam o dau. Van dang chay day du.')
else:
    print('\n-> KHONG dat. So sanh voi cot ViT: neu ca hai cung tut thi do phan giai cao KHONG hop')
    print('   voi bai nay o ngan sach hien tai, bat ke kien truc. Dong huong do phan giai.')
print('=' * 74)
print(f'CAVEAT: che do thu = E={EPOCHS}, {len(SD)} seed. Nhan hiem can nhieu buoc hon de hoi tu,')
print('        nen mot phan chenh lech co the la THIEU HUAN LUYEN chu khong phai gioi han cau truc.')
print('CHUA cham test.')


nhat ky doc tu: /content/nb41e_r1022_E8/epoch_log.json
=== 68b @ 1022px — epoch cuoi (ep8), 2 seed ===
           head  mAP_anh  medium   tail
arm                                    
cnn_full 0.5491   0.2401  0.2036 0.1004

ensemble CUNG do sau (ep<=8, seed [42, 43]): 68b@1022 16 file | 64b@518 16 file

               64b @518   68b @1022    chenh  ViT (nb68)
  macro          0.2463      0.2468  +0.0005     -0.0055
  head           0.5564      0.5541  -0.0023     -0.0048
  medium         0.2069      0.2087  +0.0018     +0.0029
  tail           0.1089      0.1097  +0.0008     -0.0138

=== 7 nhan medium ===
nhan                           64b @518   68b @1022    chenh
  Adenopathy                     0.0717      0.0677  -0.0040
  Infiltration                   0.0719      0.0698  -0.0021   <- nhan BAN, loai khoi cong
  Tortuous Aorta                 0.0748      0.0727  -0.0021
  Granuloma                      0.0952      0.1617  +0.0665
  Fissure                        0.1019      0.1005 